# Capstone 3: Intelligent Test Suite Generator

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build an **Intelligent Test Suite Generator** that analyzes an application's feature set, generates a prioritized test suite with full coverage, creates test data, estimates execution time, and produces automation-ready test scripts.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed Chapters 1-15 and Capstone 1


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Application Feature Map

Define the application features that need test coverage.


In [ ]:
# Application feature map for an e-commerce platform
app_features = {
    'application': 'ShopEasy E-Commerce Platform',
    'modules': [
        {
            'name': 'User Management',
            'features': [
                {'id': 'F-101', 'name': 'User Registration', 'complexity': 'Medium', 'risk': 'High'},
                {'id': 'F-102', 'name': 'User Login (email + social)', 'complexity': 'Medium', 'risk': 'Critical'},
                {'id': 'F-103', 'name': 'Profile Management', 'complexity': 'Low', 'risk': 'Low'},
                {'id': 'F-104', 'name': 'Password Reset', 'complexity': 'Medium', 'risk': 'High'}
            ]
        },
        {
            'name': 'Product Catalog',
            'features': [
                {'id': 'F-201', 'name': 'Product Search with Filters', 'complexity': 'High', 'risk': 'High'},
                {'id': 'F-202', 'name': 'Product Detail Page', 'complexity': 'Low', 'risk': 'Medium'},
                {'id': 'F-203', 'name': 'Product Reviews & Ratings', 'complexity': 'Medium', 'risk': 'Medium'}
            ]
        },
        {
            'name': 'Shopping Cart & Checkout',
            'features': [
                {'id': 'F-301', 'name': 'Add/Remove Cart Items', 'complexity': 'Medium', 'risk': 'Critical'},
                {'id': 'F-302', 'name': 'Apply Discount Codes', 'complexity': 'High', 'risk': 'High'},
                {'id': 'F-303', 'name': 'Checkout & Payment', 'complexity': 'High', 'risk': 'Critical'},
                {'id': 'F-304', 'name': 'Order Confirmation', 'complexity': 'Low', 'risk': 'High'}
            ]
        }
    ]
}

total_features = sum(len(m['features']) for m in app_features['modules'])
print(f"Application: {app_features['application']}")
print(f"Modules: {len(app_features['modules'])} | Features: {total_features}")

## 2. Risk-Based Test Strategy

Generate a test strategy that prioritizes testing based on risk and complexity.


In [ ]:
# Generate risk-based test strategy
strategy_prompt = f"""Create a risk-based test strategy for this application.

Application: {json.dumps(app_features)}

The strategy should:
1. Prioritize features by risk x complexity
2. Define test depth per risk level (Critical=exhaustive, High=thorough, Medium=standard, Low=smoke)
3. Specify test types per feature (functional, integration, performance, security, usability)
4. Estimate test case count per feature
5. Define automation candidates (yes/partial/no with reasoning)

Return a JSON object with:
- priority_matrix: array of {{feature_id, risk_score (1-10), test_depth, test_types, estimated_tc_count, automate}}
- total_estimated_tests: number
- testing_phases: ordered list of which features to test first

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': strategy_prompt}], temperature=0.2
)
strategy = json.loads(response.choices[0].message.content)
print(f"Total estimated test cases: {strategy['total_estimated_tests']}\n")
print('Priority Matrix:')
for item in sorted(strategy['priority_matrix'], key=lambda x: x['risk_score'], reverse=True):
    print(f"  {item['feature_id']}: Risk={item['risk_score']} | Depth={item['test_depth']} | TCs={item['estimated_tc_count']} | Automate={item['automate']}")

## 3. Generate Complete Test Suite

Create the full test suite for the highest-priority module.


In [ ]:
# Generate test suite for the highest-risk module (Shopping Cart & Checkout)
checkout_features = app_features['modules'][2]  # Shopping Cart & Checkout

suite_prompt = f"""Generate a comprehensive test suite for this module.

Module: {json.dumps(checkout_features)}
Test Strategy: {json.dumps([s for s in strategy['priority_matrix'] if s['feature_id'].startswith('F-3')])}

For each feature, generate test cases covering:
- Functional (positive and negative)
- Boundary value
- Integration (between features)
- Data validation
- Error handling

Each test case: tc_id, feature_id, title, type, priority (P1-P4), 
preconditions, steps (list), expected_result, test_data, 
automation_candidate (yes/no), estimated_minutes.

Generate at least 20 test cases total. Return as JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': suite_prompt}], temperature=0.2
)
test_suite = json.loads(response.choices[0].message.content)
print(f'Generated {len(test_suite)} test cases for {checkout_features["name"]}\n')

from collections import Counter
by_feature = Counter(tc['feature_id'] for tc in test_suite)
by_priority = Counter(tc['priority'] for tc in test_suite)
by_type = Counter(tc['type'] for tc in test_suite)

print('By Feature:', dict(by_feature))
print('By Priority:', dict(by_priority))
print('By Type:', dict(by_type))
print(f'Automation Candidates: {sum(1 for tc in test_suite if tc.get("automation_candidate") == "yes")}/{len(test_suite)}')

In [ ]:
# Generate automation scripts for automatable test cases
auto_tests = [tc for tc in test_suite if tc.get('automation_candidate') == 'yes'][:5]

script_prompt = f"""Generate Python automation scripts using pytest and Playwright for these test cases.

Test Cases: {json.dumps(auto_tests)}

Generate:
1. A pytest fixture for setup/teardown
2. Individual test functions for each test case
3. Use Page Object Model pattern
4. Include assertions for expected results
5. Add docstrings with traceability to test case IDs

Return the complete Python test file as a string."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': script_prompt}], temperature=0.2
)
print('GENERATED AUTOMATION SCRIPT:')
print('=' * 60)
print(response.choices[0].message.content)

In [ ]:
# Generate test execution schedule
total_minutes = sum(tc.get('estimated_minutes', 10) for tc in test_suite)
print(f'Total estimated execution time: {total_minutes} minutes ({total_minutes/60:.1f} hours)')
print(f'Automated tests: {sum(tc.get("estimated_minutes", 10) for tc in test_suite if tc.get("automation_candidate") == "yes")} minutes')
print(f'Manual tests: {sum(tc.get("estimated_minutes", 10) for tc in test_suite if tc.get("automation_candidate") != "yes")} minutes')

# Quick execution plan
print('\nSuggested Execution Order:')
for i, tc in enumerate(sorted(test_suite, key=lambda x: x.get('priority', 'P4')), 1):
    print(f"  {i}. [{tc['priority']}] {tc['tc_id']}: {tc['title'][:50]}")

## Exercises
1. Generate the complete test suite for all 3 modules and produce a master test plan.
2. Add cross-module integration tests (e.g., login -> search -> add to cart -> checkout).
3. Generate performance test scenarios using Locust or k6 script format.
4. Create a test suite maintenance tool that identifies obsolete tests when requirements change.
